# Clasificadores de Cocinas 🍜

En esta lección vamos a usar el dataset limpio de la lección anterior
para **entrenar modelos de clasificación** que predecirán de qué cocina
es una receta basándose en sus ingredientes.

## 1. Cargar datos limpios

Usamos `cleaned_cuisines.csv` que creamos en la lección anterior.
Ya está balanceado con SMOTE y sin los ingredientes comunes.

In [1]:
import pandas as pd

cuisines_df = pd.read_csv('../data/cleaned_cuisines.csv')
cuisines_df.head()

,Unnamed: 0,cuisine,almond,angelica,anise,anise_seed,apple,apple_brandy,apricot,armagnac,...,whiskey,white_bread,white_wine,whole_grain_wheat_flour,wine,wood,yam,yeast,yogurt,zucchini
0,0,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,indian,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,3,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,4,indian,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


## 2. Importar modelos y métricas

- **LogisticRegression**: Nuestro primer clasificador
- **SVC**: Support Vector Classifier (para comparar después)
- **Métricas**: Para evaluar qué tan bien funciona el modelo

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, confusion_matrix, classification_report, precision_recall_curve
from sklearn.svm import SVC
import numpy as np

## 3. Separar features y labels

- **Labels** (y): La columna `cuisine` — lo que queremos predecir
- **Features** (X): Todo lo demás — los ingredientes (0 o 1)

In [4]:
cuisines_label_df = cuisines_df['cuisine']
cuisines_label_df.head()

0    indian
1    indian
2    indian
3    indian
4    indian
Name: cuisine, dtype: object

In [5]:
cuisines_feature_df = cuisines_df.drop(['Unnamed: 0', 'cuisine'], axis=1)
cuisines_feature_df.head()

,almond,angelica,anise,anise_seed,apple,apple_brandy,apricot,armagnac,artemisia,artichoke,...,whiskey,white_bread,white_wine,whole_grain_wheat_flour,wine,wood,yam,yeast,yogurt,zucchini
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


## 4. Dividir en train/test

Usamos 70% para entrenar y 30% para evaluar.
Esto es clave: **nunca evalúes el modelo con los mismos datos con los que lo entrenaste**.

In [6]:
X_train, X_test, y_train, y_test = train_test_split(cuisines_feature_df, cuisines_label_df, test_size=0.3)

## 5. Entrenar Logistic Regression

Aquí es donde la magia ocurre. El modelo aprende la relación entre
ingredientes y cocinas.

- `multi_class='ovr'`: One-vs-Rest — entrena un clasificador binario por cada cocina
- `solver='liblinear'`: Algoritmo de optimización para datasets pequeños

In [7]:
lr = LogisticRegression(multi_class='ovr', solver='liblinear')
model = lr.fit(X_train, np.ravel(y_train))

accuracy = model.score(X_test, y_test)
print(f'Accuracy: {accuracy:.2%}')

Accuracy: 81.15%


c:\Users\mikel\Documents\ML-Microsoft\ML-For-Beginners\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\mikel\Documents\ML-Microsoft\ML-For-Beginners\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


## 6. Ver una predicción

Veamos qué ingredientes tiene la fila #50 del test set
y qué cocina predijo el modelo.

In [8]:
print(f'Ingredientes: {X_test.iloc[50][X_test.iloc[50]!=0].keys().tolist()}')
print(f'Cocina real: {y_test.iloc[50]}')

Ingredientes: ['black_pepper', 'cilantro', 'coconut', 'coriander', 'cumin', 'galanga', 'lemongrass', 'lime', 'soy_sauce', 'thai_pepper']
Cocina real: thai


## 7. Ver probabilidades

El modelo no solo predice la cocina — también calcula la **probabilidad**
para cada clase. Esto te dice qué tan seguro está.

In [9]:
test = X_test.iloc[50].values.reshape(-1, 1).T
proba = model.predict_proba(test)
classes = model.classes_

resultdf = pd.DataFrame(data=proba, columns=classes)
resultdf.T.sort_values(by=[0], ascending=[False]).head()

c:\Users\mikel\Documents\ML-Microsoft\ML-For-Beginners\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


,0
thai,0.984244
indian,0.012587
chinese,0.002502
japanese,0.000624
korean,0.000043


## 8. Evaluar con classification report

El reporte te muestra para **cada cocina**:
- **Precision**: De los que predije como esta cocina, ¿cuántos lo eran?
- **Recall**: De los que ERAN esta cocina, ¿cuántos detecté?
- **F1-score**: Balance entre precision y recall

In [10]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

     chinese       0.79      0.73      0.76       239
      indian       0.93      0.90      0.91       249
    japanese       0.72      0.82      0.77       242
      korean       0.84      0.75      0.79       242
        thai       0.80      0.86      0.83       227

    accuracy                           0.81      1199
   macro avg       0.81      0.81      0.81      1199
weighted avg       0.82      0.81      0.81      1199



---

## 🧪 Experimentos para entender mejor

Ahora que el modelo está entrenado, hacé estos experimentos
para realmente entender qué está pasando.

### Experimento 1: Cambiar el solver

El `solver` es el algoritmo que el modelo usa para encontrar
los mejores coeficientes. Probá dos diferentes y compará:

In [11]:
# Solver liblinear
lr_liblinear = LogisticRegression(multi_class='ovr', solver='liblinear')
model_liblinear = lr_liblinear.fit(X_train, np.ravel(y_train))
acc_liblinear = model_liblinear.score(X_test, y_test)

# Solver lbfgs
lr_lbfgs = LogisticRegression(multi_class='ovr', solver='lbfgs')
model_lbfgs = lr_lbfgs.fit(X_train, np.ravel(y_train))
acc_lbfgs = model_lbfgs.score(X_test, y_test)

print(f'liblinear: {acc_liblinear:.2%}')
print(f'lbfgs:     {acc_lbfgs:.2%}')

liblinear: 81.15%
lbfgs:     81.07%


c:\Users\mikel\Documents\ML-Microsoft\ML-For-Beginners\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\mikel\Documents\ML-Microsoft\ML-For-Beginners\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(
c:\Users\mikel\Documents\ML-Microsoft\ML-For-Beginners\.venv\lib\site-packages\sklearn\linear_model\_logistic.py:1281: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. Use OneVsRestClassifier(LogisticRegression(..)) inst

### Experimento 2: Probar otras filas

Cambiate el número `50` por otros (0, 100, 200, etc.)
y mirá si el modelo acierta o se equivoca.

In [12]:
for i in [0, 100, 200, 500]:
    ingredientes = X_test.iloc[i][X_test.iloc[i]!=0].keys().tolist()
    real = y_test.iloc[i]
    pred = model.predict(X_test.iloc[i].values.reshape(1, -1))[0]
    status = '✅' if real == pred else '❌'
    print(f'{status} Fila {i}: {real} → predicho: {pred} | ingredientes: {ingredientes[:3]}...')

❌ Fila 0: japanese → predicho: chinese | ingredientes: ['chicken', 'meat', 'soy_sauce']...
✅ Fila 100: japanese → predicho: japanese | ingredientes: ['vegetable_oil']...
✅ Fila 200: japanese → predicho: japanese | ingredientes: ['seaweed', 'vinegar']...
❌ Fila 500: korean → predicho: japanese | ingredientes: ['bell_pepper', 'fish', 'sake']...


c:\Users\mikel\Documents\ML-Microsoft\ML-For-Beginners\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mikel\Documents\ML-Microsoft\ML-For-Beginners\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mikel\Documents\ML-Microsoft\ML-For-Beginners\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\mikel\Documents\ML-Microsoft\ML-For-Beginners\.venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


### Experimento 3: Confusion Matrix

La matriz de confusión te dice **dónde se equivoca** el modelo.
Las filas son las cocinas reales, las columnas son las predichas.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=model.classes_, columns=model.classes_)
cm_df

,chinese,indian,japanese,korean,thai
chinese,174,5,27,16,17
indian,0,223,10,1,15
japanese,17,3,198,14,10
korean,20,0,33,182,7
thai,10,10,7,4,196


: 

### Experimento 4: ¿Qué cocina es más fácil de predecir?

Mirá el classification report de arriba:
- ¿Qué cocina tiene mayor **recall**? (la más fácil de detectar)
- ¿Qué cocina tiene menor **recall**? (la que más se confunde)
- ¿Por qué creés que pasa eso?

---

## ✅ Resumen

1. **Cargamos** el dataset limpio de cocinas
2. **Separamos** features (ingredientes) y labels (cuisine)
3. **Dividimos** en train (70%) y test (30%)
4. **Entrenamos** Logistic Regression con solver liblinear
5. **Evaluamos** con accuracy, classification report y confusion matrix

### Conceptos clave

| Término | Significado |
|---------|-------------|
| **Solver** | Algoritmo de optimización |
| **multi_class** | Cómo maneja más de 2 clases |
| **OvR** | One-vs-Rest: un clasificador binario por clase |
| **Accuracy** | % general de aciertos |
| **Precision** | De los predichos como X, ¿cuántos son X? |
| **Recall** | De los que son X, ¿cuántos detectó? |
| **F1-score** | Balance precision-recall |